In [2]:
import pandas as pd
print(pd.__version__)

3.0.5


In [ ]:
from sqlalchemy import create_engine, URL, text
import os

mysql_password = os.environ.get("MYSQL_PASSWORD", "")

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=mysql_password,
    host="127.0.0.1",
    port=3306,
    database="fraud_fds"
)

engine = create_engine(
    url,
    connect_args={"connect_timeout": 10},
    pool_pre_ping=True
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("연결 확인:", result.scalar())

In [4]:
import pandas as pd

df = pd.read_sql(
    """
    SELECT *
    FROM fraud_full_raw
    """,
    con=engine
)

print(df.shape)
print(df.head())

(1296675, 24)
  trans_date_trans_time            cc_num                            merchant  \
0   2019-01-01 00:00:18  2703186189652095          fraud_Rippin, Kub and Mann   
1   2019-01-01 00:00:44      630423337322     fraud_Heller, Gutmann and Zieme   
2   2019-01-01 00:00:51    38859492057661                fraud_Lind-Buckridge   
3   2019-01-01 00:01:16  3534093764340240  fraud_Kutch, Hermiston and Farrell   
4   2019-01-01 00:03:06   375534208663984                 fraud_Keeling-Crist   

        category     amt  is_fraud  recent_24h_high_amt_count  \
0       misc_net    4.97         0                          0   
1    grocery_pos  107.23         0                          0   
2  entertainment  220.11         0                          0   
3  gas_transport   45.00         0                          0   
4       misc_pos   41.96         0                          0   

   category_recent_fraud_rate  category_recent_fraud_rate_missing  \
0                         0.0          

In [5]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 trans_date_trans_time
2 cc_num
3 merchant
4 category
5 amt
6 is_fraud
7 recent_24h_high_amt_count
8 category_recent_fraud_rate
9 category_recent_fraud_rate_missing
10 count_30min
11 Repeat3
12 high_speed
13 speed_2
14 customer_mean_amt
15 customer_std_amt
16 amt_ratio_to_mean
17 amt_zscore_card
18 customer_transaction_count
19 trans_hour
20 age
21 age_group
22 prior_normal_median_amt
23 amt_to_prior_median_ratio
24 is_10x_prior_median


In [6]:
import numpy as np
import pandas as pd

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    errors="coerce"
)

df["_row_order"] = np.arange(len(df))

df = (
    df.sort_values(
        ["cc_num", "trans_date_trans_time", "_row_order"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

df["_normal_amt"] = df["amt"].where(df["is_fraud"] == 0)

df["prior_normal_median_amt"] = (
    df.groupby("cc_num", sort=False)["_normal_amt"]
      .transform(
          lambda x: x.shift(1)
                     .expanding(min_periods=1)
                     .median()
      )
)

In [7]:
print(
    df[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "is_fraud",
            "prior_normal_median_amt"
        ]
    ].head(20)
)

print(
    "결측치 수:",
    df["prior_normal_median_amt"].isna().sum()
)

         cc_num trans_date_trans_time     amt  is_fraud  \
0   60416207185   2019-01-01 12:47:15    7.27         0   
1   60416207185   2019-01-02 08:44:57   52.94         0   
2   60416207185   2019-01-02 08:47:36   82.08         0   
3   60416207185   2019-01-02 12:38:14   34.79         0   
4   60416207185   2019-01-02 13:10:46   27.18         0   
5   60416207185   2019-01-03 13:56:35    6.87         0   
6   60416207185   2019-01-03 17:05:10    8.43         0   
7   60416207185   2019-01-04 13:59:55  117.11         0   
8   60416207185   2019-01-04 21:17:22   26.74         0   
9   60416207185   2019-01-05 00:42:24  105.20         0   
10  60416207185   2019-01-05 21:34:20    4.98         0   
11  60416207185   2019-01-06 10:25:49  102.47         0   
12  60416207185   2019-01-07 12:58:19  204.15         0   
13  60416207185   2019-01-08 08:05:23   64.31         0   
14  60416207185   2019-01-08 23:20:22  200.77         0   
15  60416207185   2019-01-08 23:24:43   81.48         0 

In [8]:
df["amt_to_prior_median_ratio"] = (
    df["amt"] / df["prior_normal_median_amt"]
)

In [9]:
print(
    df[
        [
            "amt",
            "prior_normal_median_amt",
            "amt_to_prior_median_ratio"
        ]
    ].head(20)
)

print(
    "비율 결측치 수:",
    df["amt_to_prior_median_ratio"].isna().sum()
)

       amt  prior_normal_median_amt  amt_to_prior_median_ratio
0     7.27                      NaN                        NaN
1    52.94                    7.270                   7.281981
2    82.08                   30.105                   2.726457
3    34.79                   52.940                   0.657159
4    27.18                   43.865                   0.619628
5     6.87                   34.790                   0.197471
6     8.43                   30.985                   0.272067
7   117.11                   27.180                   4.308683
8    26.74                   30.985                   0.862998
9   105.20                   27.180                   3.870493
10    4.98                   30.985                   0.160723
11  102.47                   27.180                   3.770052
12  204.15                   30.985                   6.588672
13   64.31                   34.790                   1.848520
14  200.77                   43.865                   4

In [10]:
df["is_10x_prior_median"] = (
    df["prior_normal_median_amt"].notna()
    & (df["prior_normal_median_amt"] > 0)
    & (df["amt_to_prior_median_ratio"] >= 10)
).astype("int8")

In [11]:
print(df["is_10x_prior_median"].value_counts(dropna=False).sort_index())

is_10x_prior_median
0    1279576
1      17099
Name: count, dtype: int64


In [ ]:
## 만들어진 3개 변수를 넣기 전에 임시 변수 제거(단, _row_order는 아직 삭제 x. sql의 원래 행과 정확히 맞춰 넣으려면 다음 단계에서 필요.)
df.drop(
    columns=["_normal_amt"],
    inplace=True
)

In [13]:
df = (
    df.sort_values("_row_order")
      .reset_index(drop=True)
)

df.drop(columns=["_row_order"], inplace=True)

print(df.shape)

(1296675, 24)


In [14]:
key_cols = [
    "cc_num",
    "trans_date_trans_time",
    "merchant",
    "amt"
]

duplicate_count = df.duplicated(
    subset=key_cols,
    keep=False
).sum()

print("중복 거래 행 수:", duplicate_count)

중복 거래 행 수: 0


In [15]:
import numpy as np

df["_row_order"] = np.arange(len(df))

df = (
    df.sort_values(
        ["cc_num", "trans_date_trans_time", "_row_order"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

df["_is_normal"] = (df["is_fraud"] == 0).astype("int8")

prior_normal_count = (
    df.groupby("cc_num", sort=False)["_is_normal"]
      .cumsum()
    - df["_is_normal"]
)

df["has_prior_normal_transaction"] = (
    prior_normal_count > 0
).astype("int8")

In [16]:
print(
    df["has_prior_normal_transaction"]
    .value_counts()
    .sort_index()
)

has_prior_normal_transaction
0       1649
1    1295026
Name: count, dtype: int64


이 1,649건이 아까 prior_normal_median_amt, amt_to_prior_median_ratio의 결측치 수와 정확히 같아서 계산도 일관됨.

In [18]:
import numpy as np
from numba import njit

@njit
def calculate_outside_trans_hours_80(
    cc_nums,
    hours,
    normal_flags,
    min_prior_normal=20,
    coverage=0.80
):
    n = len(cc_nums)
    result = np.zeros(n, dtype=np.int8)

    hour_counts = np.zeros(24, dtype=np.int64)
    prior_normal_total = 0
    current_cc = cc_nums[0]

    for i in range(n):
        # 새로운 고객이면 이력 초기화
        if cc_nums[i] != current_cc:
            current_cc = cc_nums[i]
            hour_counts[:] = 0
            prior_normal_total = 0

        current_hour = hours[i]

        # 현재 거래보다 앞선 정상거래만 사용
        if prior_normal_total >= min_prior_normal:
            sorted_counts = np.sort(hour_counts)[::-1]
            target = prior_normal_total * coverage

            cumulative = 0
            cutoff_count = 0

            for j in range(24):
                cumulative += sorted_counts[j]

                if cumulative >= target:
                    cutoff_count = sorted_counts[j]
                    break

            if hour_counts[current_hour] < cutoff_count:
                result[i] = 1

        # 현재 거래가 정상이면 평가 후 이력에 추가
        if normal_flags[i] == 1:
            hour_counts[current_hour] += 1
            prior_normal_total += 1

    return result


cc_array = df["cc_num"].to_numpy(dtype=np.int64)
hour_array = df["trans_date_trans_time"].dt.hour.to_numpy(dtype=np.int8)
normal_array = df["_is_normal"].to_numpy(dtype=np.int8)

df["outside_trans_hours_80"] = calculate_outside_trans_hours_80(
    cc_array,
    hour_array,
    normal_array
)

print(
    df["outside_trans_hours_80"]
    .value_counts()
    .sort_index()
)

outside_trans_hours_80
0    1028541
1     268134
Name: count, dtype: int64


In [19]:
check = pd.crosstab(
    prior_normal_count >= 20,
    df["outside_trans_hours_80"]
)

print(check)

outside_trans_hours_80        0       1
_is_normal                             
False                     19099       0
True                    1009442  268134


In [20]:
online_categories = [
    "shopping_net",
    "misc_net",
    "grocery_net"
]

df["is_online"] = (
    df["category"]
    .isin(online_categories)
    .astype("int8")
)

print(
    df["is_online"]
    .value_counts()
    .sort_index()
)

is_online
0    1090393
1     206282
Name: count, dtype: int64


In [21]:
df["risk_time_22_04"] = (
    df["trans_date_trans_time"]
    .dt.hour
    .isin([22, 23, 0, 1, 2, 3])
    .astype("int8")
)

print(
    df["risk_time_22_04"]
    .value_counts()
    .sort_index()
)

risk_time_22_04
0    991793
1    304882
Name: count, dtype: int64


In [22]:
df["interact_repeat_category"] = (
    df["recent_24h_high_amt_count"]
    * df["category_recent_fraud_rate"]
)

print(
    df["interact_repeat_category"]
    .describe()
)

print(
    "결측치 수:",
    df["interact_repeat_category"].isna().sum()
)

count    1.296675e+06
mean     3.315781e-04
std      3.287061e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.056027e-01
Name: interact_repeat_category, dtype: float64
결측치 수: 0


In [23]:
df = (
    df.sort_values(
        ["cc_num", "trans_date_trans_time", "_row_order"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

df["_prev_merchant"] = (
    df.groupby("cc_num", sort=False)["merchant"]
      .shift(1)
)

df["merchant_change_count"] = (
    df["_prev_merchant"].notna()
    & (df["merchant"] != df["_prev_merchant"])
).astype("int8")

df.drop(columns=["_prev_merchant"], inplace=True)

print(
    df["merchant_change_count"]
    .value_counts()
    .sort_index()
)

merchant_change_count
0       3644
1    1293031
Name: count, dtype: int64


In [ ]:
##

rolling_sum = (
    df.set_index("trans_date_trans_time")
      .groupby("cc_num", sort=False)["amt"]
      .rolling("1h", closed="both")
      .sum()
      .reset_index(
          level=0,
          drop=True
      )
)

df["rolling_sum_amt_1h"] = (
    rolling_sum
    .reset_index(drop=True)
    .astype("float64")
)

print(df["rolling_sum_amt_1h"].describe())
print("결측치 수:", df["rolling_sum_amt_1h"].isna().sum())

count    1.296675e+06
mean     8.491117e+01
std      1.947081e+02
min      1.000000e+00
25%      1.425000e+01
50%      5.429000e+01
75%      9.630000e+01
max      2.894890e+04
Name: rolling_sum_amt_1h, dtype: float64
결측치 수: 0


In [26]:
import numpy as np
import pandas as pd

# 1. 계산용 임시 컬럼 제거
temp_cols = [
    "_row_order",
    "_is_normal"
]

df_sql = df.drop(
    columns=[col for col in temp_cols if col in df.columns]
).copy()

# 2. MySQL에 저장할 수 있도록 무한대 값을 NULL로 변환
df_sql.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

# 3. 적재 전 확인
print("적재할 행·열 수:", df_sql.shape)
print("임시 컬럼 잔존 여부:", [
    col for col in df_sql.columns
    if col.startswith("_")
])

# 4. 새 SQL 테이블에 전체 적재
df_sql.to_sql(
    name="fraud_full_features",
    con=engine,
    schema="fraud_fds",
    if_exists="replace",
    index=False,
    chunksize=5000,
    method=None
)

print("SQL 적재 완료")

적재할 행·열 수: (1296675, 31)
임시 컬럼 잔존 여부: []
SQL 적재 완료


In [27]:
sql_check = pd.read_sql(
    """
    SELECT COUNT(*) AS row_count
    FROM fraud_fds.fraud_full_features
    """,
    con=engine
)

column_check = pd.read_sql(
    """
    SELECT COUNT(*) AS column_count
    FROM information_schema.columns
    WHERE table_schema = 'fraud_fds'
      AND table_name = 'fraud_full_features'
    """,
    con=engine
)

print(sql_check)
print(column_check)

   row_count
0    1296675
   column_count
0            31


In [28]:
csv_path = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

df_sql.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV 저장 완료:", csv_path)
print("행·열 수:", df_sql.shape)

CSV 저장 완료: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\fraud_full_features.csv
행·열 수: (1296675, 31)


In [29]:
# age_group 삭제
df_sql.drop(
    columns=["age_group"],
    inplace=True,
    errors="ignore"
)

# 현재 거래금액이 500달러 이상이면 1
df_sql["is_high_amt"] = (
    df_sql["amt"] >= 500
).astype("int8")

print(
    df_sql["is_high_amt"]
    .value_counts()
    .sort_index()
)

print("현재 행·열 수:", df_sql.shape)

is_high_amt
0    1281044
1      15631
Name: count, dtype: int64
현재 행·열 수: (1296675, 31)


In [30]:
df_sql["high_speed"] = (
    df_sql["high_speed"] >= 100
).astype("int8")

print(
    df_sql["high_speed"]
    .value_counts(dropna=False)
    .sort_index()
)

print("자료형:", df_sql["high_speed"].dtype)

high_speed
0    1139777
1     156898
Name: count, dtype: int64
자료형: int8


In [32]:
csv_path = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

df_sql.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV 갱신 완료:", csv_path)
print("행·열 수:", df_sql.shape)
print("age_group 존재 여부:", "age_group" in df_sql.columns)
print("is_high_amt 존재 여부:", "is_high_amt" in df_sql.columns)
print("high_speed 자료형:", df_sql["high_speed"].dtype)

CSV 갱신 완료: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\fraud_full_features.csv
행·열 수: (1296675, 31)
age_group 존재 여부: False
is_high_amt 존재 여부: True
high_speed 자료형: int8


# SQL에 적재

In [33]:
df_sql.to_sql(
    name="fraud_full_features",
    con=engine,
    schema="fraud_fds",
    if_exists="replace",
    index=False,
    chunksize=5000,
    method=None
)

print("MySQL 적재 완료")

MySQL 적재 완료


In [34]:
sql_check = pd.read_sql(
    """
    SELECT COUNT(*) AS row_count
    FROM fraud_fds.fraud_full_features
    """,
    con=engine
)

column_check = pd.read_sql(
    """
    SELECT COUNT(*) AS column_count
    FROM information_schema.columns
    WHERE table_schema = 'fraud_fds'
      AND table_name = 'fraud_full_features'
    """,
    con=engine
)

variable_check = pd.read_sql(
    """
    SELECT
        SUM(is_high_amt = 1) AS high_amt_1_count,
        SUM(high_speed = 1) AS high_speed_1_count
    FROM fraud_fds.fraud_full_features
    """,
    con=engine
)

print(sql_check)
print(column_check)
print(variable_check)

   row_count
0    1296675
   column_count
0            31
   high_amt_1_count  high_speed_1_count
0           15631.0            156898.0
